---
last_verified: 2026-10-07
tool_version: n/a
sources: []
---

# Choosing between CLI and API modes in Snyk

> Reference notebook. Release-neutral by design: no Snyk version is claimed here, and flag names should be taken from the installed binary's help output, not from memory. No external sources are cited because the current research cycle covers other tools; all cross-references below point at files already in this kit.

## Purpose

Every Snyk integration eventually asks the same question: should this scan run through the Snyk CLI, or through the HTTP API from custom code? Both surfaces drive the same scan engine, so the finding set for identical inputs should agree — but they differ in who can call them, how results are consumed, and how failures surface. This notebook records a decision rule for picking one mode per integration, plus runnable checks that encode the rule so a pipeline can apply it instead of re-deciding it on every job.

The two modes under comparison:

- **CLI mode** — invoke `snyk test` (or `snyk container test` for container images) from a shell step, authenticate with `SNYK_TOKEN` or a one-time `snyk auth` session, and read the exit code plus terminal output (or `--json` for machine-readable results). This is the shape `snyk/scripts/snyk-vuln-scan-pipeline.sh` and `snyk/manifests/snyk-github-actions-cicd-workflow.yaml` already use.
- **API mode** — call the Snyk HTTP API from application code with a bearer token, receive JSON responses, and shape findings inside the consumer. No CLI install is required on the caller.

## When to use

- A stock CI job needs a gate that fails the build on findings above a threshold.
- A custom platform must ingest findings, correlate them with its own data, or batch many projects.
- Scans run in an environment where the runtime image may be built offline (see the air-gapped CLI image) and the caller still needs deterministic behavior.
- A team keeps rewriting shell wrappers around `--json` output and the wrapper itself is becoming the maintenance burden.

## Prerequisites

- A Snyk account and an API token, kept out of committed files; the kit convention is the `SNYK_TOKEN` environment variable or CI secret.
- The Snyk CLI on `PATH` for CLI mode; the kit installs it with `npm install -g snyk` in CI, and `snyk/dockerfiles/custom-snyk-cli-air-gapped.Dockerfile` shows the pre-bundled form for offline runtimes.
- Python 3 with only the standard library (`json`) for the cells below — every code cell runs offline against embedded fixtures.
- The stage layout in `snyk/docs/integrating-snyk-with-cicd-security-scanning.md` as the pipeline context this decision slots into.


In [ ]:
"""Decision helper: recommend a Snyk integration mode from caller requirements.

The rule mirrors the kit's own integrations: stock CI gates and terminal runs
use the CLI; custom code that must shape findings as data uses the API;
offline runtimes can only pre-bundle the CLI.
"""


def recommend_mode(*, caller, output_consumer, environment, needs_dashboard_history=False):
    """Return (mode, rationale) for one integration.

    caller:                  "human" | "ci" | "custom-app"
    output_consumer:         "terminal" | "ci-gate" | "programmatic"
    environment:             "networked" | "air-gapped"
    needs_dashboard_history: results must accumulate on the Snyk dashboard
    """
    if environment == "air-gapped":
        return (
            "cli",
            "Offline runtime: only the CLI can be pre-bundled into the image "
            "and run without caller-side egress.",
        )
    if caller == "custom-app" or output_consumer == "programmatic":
        return (
            "api",
            "Consumer is custom code: the API returns findings as JSON that the "
            "application can batch, correlate, and reshape without shell wrappers.",
        )
    mode = "cli"
    rationale = (
        "Consumer is a human terminal or a stock CI step: the CLI's exit code "
        "is the gate and `--json` covers the machine-readable case."
    )
    if needs_dashboard_history:
        rationale += (
            " Pair the scan with `snyk monitor` so snapshots accumulate on the "
            "dashboard under a stable project name."
        )
    return (mode, rationale)


# Scenario matrix: (caller, output_consumer, environment, history) -> expected mode.
scenarios = [
    (("human", "terminal", "networked", False), "cli"),
    (("ci", "ci-gate", "networked", True), "cli"),
    (("ci", "ci-gate", "air-gapped", False), "cli"),
    (("custom-app", "programmatic", "networked", False), "api"),
    (("custom-app", "programmatic", "networked", True), "api"),
]

for (caller, consumer, env, history), expected in scenarios:
    mode, rationale = recommend_mode(
        caller=caller,
        output_consumer=consumer,
        environment=env,
        needs_dashboard_history=history,
    )
    status = "OK" if mode == expected else "MISMATCH"
    print(f"[{status}] caller={caller} consumer={consumer} env={env} history={history}")
    print(f"         -> {mode}: {rationale}")
    assert mode == expected, f"decision drift for {(caller, consumer, env, history)}"

print(f"\nAll {len(scenarios)} scenarios match the decision rule.")


## Step 1: compare the modes on the deciding criteria

| Criterion | CLI mode | API mode |
|---|---|---|
| Caller | humans, CI jobs, shell scripts | custom applications and platforms |
| Output contract | terminal tables by default; `--json` for machine-readable reports | JSON over HTTPS, shaped by the caller |
| Failure surface | process exit code: non-zero when findings exceed the threshold or the scan errors | HTTP status code plus a JSON error body |
| Authentication | `SNYK_TOKEN` in the environment, or a one-time `snyk auth` session | bearer token attached to every request |
| Network at runtime | the scan calls the Snyk service; the binary itself can be pre-bundled for offline images | every call needs outbound HTTPS from the caller |
| Install footprint | CLI on the runner or in the image | none — any HTTP client suffices |
| Command breadth | subcommands for `test`, `monitor`, `container test`, and IaC scans | programmatic access to scan results and project data |
| Throughput shape | one process per scan, sequential by default | request-per-call; per-account limits apply, so batch with care |

The CLI's machine-readable contract is the same finding shape a custom consumer would fetch through the API: the kit's pipeline script extracts `.vulnerabilities[]` entries carrying `id`, `title`, and `severity` from `snyk test --json` output. A consumer switching modes should validate that the fields it reads exist in both shapes before migrating a gate.


In [ ]:
# Synthetic fixture in the report shape the kit's pipeline script consumes:
# a top-level "vulnerabilities" array whose entries carry id, title, severity.
# The IDs are synthetic (SYNTH-*) and stand in for real advisory identifiers;
# they exercise the counting code, not the content of any advisory.
import json

synthetic_report = {
    "vulnerabilities": [
        {"id": "SYNTH-0001", "title": "Synthetic finding A", "severity": "critical"},
        {"id": "SYNTH-0002", "title": "Synthetic finding B", "severity": "high"},
        {"id": "SYNTH-0003", "title": "Synthetic finding C", "severity": "high"},
        {"id": "SYNTH-0004", "title": "Synthetic finding D", "severity": "medium"},
    ]
}


def severity_counts(report):
    """Count findings per severity, mirroring the jq aggregation in
    snyk/scripts/snyk-vuln-scan-pipeline.sh."""
    counts = {}
    for finding in report.get("vulnerabilities", []):
        severity = finding.get("severity", "unknown")
        counts[severity] = counts.get(severity, 0) + 1
    return counts


counts = severity_counts(synthetic_report)
assert counts == {"critical": 1, "high": 2, "medium": 1}, counts
print(json.dumps(counts, indent=2))


def breaches_threshold(counts, threshold):
    """A gate at the given severity fails when any finding at or above it exists."""
    order = ["low", "medium", "high", "critical"]
    floor = order.index(threshold)
    return any(order.index(severity) >= floor for severity in counts)


assert breaches_threshold(counts, "high") is True
assert breaches_threshold({"medium": 4}, "high") is False
print("threshold gate verified against the fixture counts")


## Step 2: decide how failures surface

The mode determines what "the scan failed" means to the caller:

- **CLI mode** — the process exit code carries the verdict. A non-zero exit means either findings above the threshold or an execution error (authentication, missing manifest, network), and the caller must treat both as failure. The kit's pipeline helper does exactly this: it captures the report regardless of exit status, then exits non-zero when severity counts breach the threshold, and the CI integration doc calls out swallowing the exit status as a common error.
- **API mode** — the HTTP status code carries the verdict. A successful response with a findings payload is a scan that ran and may still contain findings; the caller decides what the payload means. An unreachable endpoint or an error status must fail closed — an API integration that treats a transport error as "no findings" reports a clean scan that never ran.

Decision rule: if the caller can only react to "process failed or not", use the CLI. If the caller needs to distinguish "clean", "findings present", and "scan did not run" as three separate states, the API's status-plus-payload split is the better fit — provided the consumer implements the fail-closed path.


## Step 3: record the decision per integration

Whichever mode wins, write the choice down where the next maintainer looks: the workflow or service that owns the scan. A minimal decision record names the caller, the output consumer, the environment, the chosen mode, and the failure semantics the caller implements. The kit's CI workflow is a worked example: the pull-request job gates with `snyk test --severity-threshold=medium --fail-on=upgradable` (exit-code semantics), while the default-branch and scheduled jobs record snapshots with `snyk monitor` (no gate). Two jobs in one repository, two different outcomes, each chosen deliberately.

When a repository later migrates one job from CLI to API (or back), the decision record is what makes the change reviewable: the finding contract, the failure semantics, and the auth model all move together, and the severity-counting logic above is the piece that must keep agreeing across the migration.


## Verify

1. For a CLI-mode gate, run the scan helper (`snyk/scripts/snyk-vuln-scan-pipeline.sh`) against a scratch target in a test branch and confirm a non-zero exit when findings breach the configured threshold, and exit zero below it.
2. For an API-mode consumer, fetch one project's findings and confirm the payload exposes the same fields the CLI's `--json` contract provides (`id`, `title`, `severity`) before any gate logic depends on them.
3. In an offline build environment, build `snyk/dockerfiles/custom-snyk-cli-air-gapped.Dockerfile` and confirm the CLI is present in the image without a runtime install step; note the scan itself still calls the Snyk service, so this image removes the install-time egress dependency, not the scan-time one.
4. Confirm the two failure states are distinguishable in the chosen mode: an execution error is never reported as a clean scan, and the error path is tested deliberately (for example, by removing the token in a test branch).
5. Confirm the decision record names the caller, consumer, environment, mode, and failure semantics, and that it lives with the workflow that owns the scan.


## Common errors

1. **Choosing the API for a stock CI gate.** The extra plumbing buys nothing over `snyk test --fail-on=<threshold>` when the consumer is a CI step that only needs the exit code.
2. **Failing open on transport errors.** An API consumer that treats an unreachable endpoint as "no findings" reports a clean scan that never ran. Fix: fail closed on any error status or connection failure.
3. **Treating a non-zero CLI exit as only "findings".** The same exit code covers execution errors (authentication, missing manifest, network). Fix: check the error output separately from the severity counts, the way the pipeline helper's summary does.
4. **Mixed modes without a recorded reason.** One job gates via the CLI while another fetches the same project via the API, and nobody can say why both exist. Fix: one decision record per integration, reviewed when either job changes.
5. **Hard-coding the token.** Both modes accept the credential from the environment or secret store; the kit convention is `SNYK_TOKEN` as a CI secret. A token in a committed command line is a finding in itself.


## References

- `snyk/scripts/snyk-vuln-scan-pipeline.sh` — CLI mode wired into a pipeline: `snyk test --json`, severity-count aggregation, threshold exit, and `snyk monitor` for dashboard history.
- `snyk/manifests/snyk-github-actions-cicd-workflow.yaml` — CLI jobs in CI: pull-request `snyk test` gate plus default-branch and scheduled `snyk monitor` snapshots.
- `snyk/configs/snyk-ci-github-actions.yaml` — CLI install and `--all-projects --fail-on=high --json` scan configuration for CI.
- `snyk/dockerfiles/custom-snyk-cli-air-gapped.Dockerfile` — pre-bundled CLI image for offline runtimes.
- `snyk/docs/integrating-snyk-with-cicd-security-scanning.md` — the stage layout (gated test vs recorded monitor) this decision slots into.
- `snyk/notes/0000-primer-snyk.md` — first-contact primer for Snyk.
